# 🖼️ ImageForge — PNG to JPG Converter

Batch image converter untuk mengubah file **PNG → JPG/JPEG** secara otomatis.

Dirancang untuk workflow batch processing sebelum proses **image upscaling** atau **AI pipeline**.

---

## ✨ Features

- 📂 Batch PNG conversion
- 🎯 Preserve filename
- ⚙️ Adjustable JPEG quality
- 📦 Automatic output folder
- 🚀 Fast processing

### 📦 Instalasi Dependensi
Sel ini memastikan semua pustaka Python yang diperlukan, terutama `gradio` untuk antarmuka pengguna, telah terinstal di lingkungan Colab.

In [24]:
# run once in firts time
!pip install gradio

### 🛠️ Pengaturan Lingkungan dan Fungsi Inti
Bagian ini mendefinisikan direktori kerja (workspace), fungsi-fungsi utama untuk konversi gambar, dan antarmuka pengguna Gradio.

#### Variabel Lingkungan & Direktori
Variabel-variabel ini menetapkan lokasi absolut untuk penyimpanan file sementara dan output hasil konversi.

In [25]:
# @title
# ==========================================
# ImageForge - Batch Image Converter (Gradio)
# ==========================================
# Skrip ini dirancang untuk dijalankan di Google Colab.
# Pastikan Anda telah menginstal Gradio: !pip install gradio

import os
import zipfile
import tempfile
import shutil
import subprocess
import threading
import re
import socket
import queue
from PIL import Image
import gradio as gr

# Setup Workspace dengan Absolute Path (Sangat penting untuk Colab Environment)
WORKSPACE_DIR = os.path.abspath("imageforge_workspace")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "output")
ZIP_PATH = os.path.join(WORKSPACE_DIR, "imageforge_output.zip")

#### `process_single_image(image_path, target_format, quality_val, output_dir)`
Fungsi ini bertanggung jawab untuk mengonversi satu gambar dari format aslinya ke format target (JPG atau PNG). Ini secara khusus menangani gambar PNG dengan latar belakang transparan (alpha channel) saat dikonversi ke JPG, dengan mengisi latar belakang putih secara default.

In [26]:
def process_single_image(image_path, target_format, quality_val, output_dir):
    """
    Fungsi inti untuk mengkonversi satu gambar.
    Menangani transparansi (alpha channel) saat konversi ke JPG.
    """
    try:
        with Image.open(image_path) as img:
            # Pastikan ekstensi huruf kecil untuk pengecekan
            target_ext = target_format.lower()

            # Konversi PNG (RGBA) ke JPG (RGB)
            if target_ext in ['jpg', 'jpeg'] and img.mode in ('RGBA', 'LA'):
                # Buat background putih
                background = Image.new('RGB', img.size, (255, 255, 255))
                # Paste gambar asli di atas background putih menggunakan alpha channel sebagai mask
                if len(img.split()) == 4: # RGBA
                    background.paste(img, mask=img.split()[3])
                else: # LA (Grayscale with Alpha)
                    background.paste(img, mask=img.split()[1])
                img = background
            elif target_ext in ['jpg', 'jpeg'] and img.mode != 'RGB':
                # Konversi mode lain (seperti P) ke RGB sebelum menyimpan sebagai JPG
                img = img.convert('RGB')

            # Ambil nama file tanpa ekstensi asli
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            output_path = os.path.join(output_dir, f"{base_name}.{target_ext}")

            if target_ext in ['jpg', 'jpeg']:
                img.save(output_path, 'JPEG', quality=quality_val)
            else:
                img.save(output_path, 'PNG')

            return output_path
    except Exception as e:
        print(f"Error memproses gambar: {e}")
        return None

#### `create_zip(source_dir, zip_path)`
Fungsi utilitas ini mengompres semua file hasil konversi dari `source_dir` ke dalam satu file ZIP di `zip_path`. Ini digunakan untuk mengunduh banyak gambar sekaligus.

In [27]:
def create_zip(source_dir, zip_path):
    """Membungkus direktori hasil ke dalam file ZIP."""
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(source_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Simpan di zip tanpa path direktori penuh
                zipf.write(file_path, arcname=file)
    return zip_path

#### `convert_images(input_files, target_format, quality_val, progress)`
Ini adalah fungsi utama yang dihubungkan dengan antarmuka Gradio. Ini menerima daftar file gambar yang diunggah, format target, dan nilai kualitas (untuk JPG), lalu memanggil `process_single_image` untuk setiap file. Fungsi ini juga menangani pembuatan file ZIP jika ada lebih dari satu gambar yang diproses dan menampilkan pratinjau gambar pertama.

In [28]:
def convert_images(input_files, target_format, quality_val, progress=gr.Progress()):
    """
    Fungsi utama yang dipanggil oleh Gradio.
    Menangani pemrosesan batch dari beberapa file yang diunggah.
    """
    if not input_files:
        return None, "Silakan unggah setidaknya satu gambar.", None

    # Bersihkan dan siapkan folder workspace
    if os.path.exists(WORKSPACE_DIR):
        shutil.rmtree(WORKSPACE_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    processed_files = []

    # Menggunakan progress bar bawaan Gradio untuk UI yang responsif
    for i, file_obj in progress.tqdm(enumerate(input_files), total=len(input_files), desc="Memproses Gambar..."):
        file_path = file_obj if isinstance(file_obj, str) else file_obj.name

        output_path = process_single_image(file_path, target_format, quality_val, OUTPUT_DIR)

        if output_path:
            processed_files.append(output_path)

    if not processed_files:
        return None, "❌ Gagal mengkonversi gambar. Periksa format file.", None

    # Trik Colab: Ekstrak gambar pertama menggunakan PIL Object secara langsung
    # agar tidak diblokir oleh proxy Colab saat dikirim ke UI
    first_image_pil = None
    try:
        first_image_pil = Image.open(processed_files[0])
    except:
        pass

    # Jika hanya 1 file, kembalikan file tersebut. Jika batch, kembalikan ZIP.
    if len(processed_files) == 1:
        result_file = processed_files[0]
        message = f"✅ Berhasil mengkonversi 1 gambar ke {target_format}."
    else:
        result_file = create_zip(OUTPUT_DIR, ZIP_PATH)
        message = f"✅ Berhasil mengkonversi {len(processed_files)} gambar. Mengunduh sebagai ZIP."

    return result_file, message, first_image_pil

### 🌐 Antarmuka Gradio (UI/UX)
Bagian ini mendefinisikan tata letak dan komponen antarmuka pengguna (`gr.Blocks`) menggunakan pustaka Gradio. Ini mencakup elemen untuk mengunggah file, memilih format target, mengatur kualitas JPG, tombol konversi, area output untuk mengunduh hasil, status progres, dan pratinjau gambar.

In [29]:
# ==========================================
# Antarmuka Gradio (UI/UX)
# ==========================================

# Mengembalikan Tema Soft bawaan tanpa custom CSS agar tampilan tetap di tengah (center)
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as demo:
    gr.Markdown(
        """
        # 🔨 ImageForge
        **Batch Image Converter for AI Pipelines**
        Unggah gambar PNG atau JPG Anda, tentukan format target, dan ImageForge akan memprosesnya.
        Sempurna untuk merapikan dataset (otomatis mengubah background transparan PNG menjadi putih saat menjadi JPG).
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            # Input area
            input_images = gr.File(
                label="Unggah Gambar (Bisa pilih banyak file sekaligus)",
                file_count="multiple",
                file_types=["image"],
                type="filepath" # Mengembalikan path file, lebih aman untuk memory
            )

            with gr.Group():
                target_format = gr.Radio(
                    choices=["JPG", "PNG"],
                    value="JPG",
                    label="Format Target",
                    info="Pilih format output yang diinginkan."
                )

                # Slider kualitas (hanya berpengaruh untuk JPG)
                quality_slider = gr.Slider(
                    minimum=10,
                    maximum=100,
                    value=95,
                    step=1,
                    label="Kualitas JPG (10-100)",
                    info="Kualitas kompresi. 95 sangat disarankan sebelum proses upscaling."
                )

            convert_btn = gr.Button("Mulai Konversi 🚀", variant="primary")

        with gr.Column(scale=1):
            # Output area
            output_file = gr.File(label="Hasil Konversi (Unduh di sini)")
            status_text = gr.Textbox(label="Status Progres", interactive=False, lines=2)

            gr.Markdown("### Preview Hasil (Gambar Pertama)")
            # Menggunakan type="pil" agar gambar ditampilkan dari memori (Bypass Colab Proxy)
            preview_image = gr.Image(label="Preview", type="pil", interactive=False)

    # Event listener
    convert_btn.click(
        fn=convert_images,
        inputs=[input_images, target_format, quality_slider],
        outputs=[output_file, status_text, preview_image]
    )

/tmp/ipykernel_19493/1676036990.py:6: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as demo:


### 🚀 Setup Tunneling (Cloudflare)
Karena Colab berjalan di lingkungan virtual, akses langsung ke server lokal sering kali dibatasi. Bagian ini menyediakan fungsionalitas tunneling menggunakan `cloudflared` untuk membuat tautan publik yang dapat diakses, memungkinkan pengguna untuk berinteraksi dengan antarmuka Gradio yang berjalan di Colab seolah-olah itu adalah aplikasi web biasa.

*   `run_cloudflared(port, url_q)`: Fungsi ini mengunduh dan menjalankan biner `cloudflared`, membuat terowongan aman ke port lokal yang ditentukan, dan mengirimkan URL publik yang dihasilkan ke antrean.
*   `get_free_port()`: Fungsi pembantu untuk menemukan port TCP yang tersedia secara dinamis agar menghindari konflik port.

In [30]:
# ==========================================
# Tunneling Setup (MENGGUNAKAN CLOUDFLARE)
# ==========================================
def run_cloudflared(port, url_q):
    """Menjalankan Cloudflare Tunnel di background dan mengirimkan URL ke antrean"""
    print("\n⏳ Menghubungkan ke server Cloudflare (Mohon tunggu sekitar 5 detik)...")
    if not os.path.exists('./cloudflared'):
        subprocess.run(['wget', '-q', '-O', 'cloudflared', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
        subprocess.run(['chmod', '+x', './cloudflared'])

    cmd = ['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}']
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in process.stdout:
        # Menggunakan regex untuk menangkap full URL
        match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            url_q.put(url)
            break

def get_free_port():
    """Mencari port kosong yang tersedia di sistem untuk menghindari error 'port in use'."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

### ▶️ Menjalankan Aplikasi ImageForge
Sel ini adalah titik masuk utama untuk menjalankan aplikasi. Ini menginisialisasi proses tunneling Cloudflare di *background*, menunggu URL publik dihasilkan, lalu meluncurkan antarmuka Gradio pada port lokal yang bebas. URL Cloudflare yang dihasilkan kemudian dicetak ke konsol agar pengguna dapat mengakses aplikasi.

In [31]:
# @title
# Jalankan aplikasi.
if __name__ == "__main__":
    url_q = queue.Queue()
    FREE_PORT = get_free_port()

    # 1. Jalankan Cloudflare duluan
    tunnel_thread = threading.Thread(target=run_cloudflared, args=(FREE_PORT, url_q))
    tunnel_thread.daemon = True
    tunnel_thread.start()

    # 2. TAHAN PROSES DI SINI sampai Cloudflare memberikan Link-nya
    cloudflare_url = url_q.get()

    # 3. Cetak Link secara dominan agar user fokus ke link ini
    print("\n" + "🔵" * 30)
    print(f"🚀 MESIN IMAGEFORGE BERHASIL MENYALA!")
    print(f"👉 KLIK LINK INI UNTUK MEMBUKA APLIKASI: {cloudflare_url}")
    print("🔵" * 30 + "\n")

    # 4. Launch Gradio lokal (Mesin Utama)
    # inline=False memastikan UI tidak tampil menyesakkan di dalam Colab
    demo.launch(
        server_name="127.0.0.1",
        server_port=FREE_PORT,
        share=False,
        debug=False,
        inline=False,
        allowed_paths=[WORKSPACE_DIR] # Menggunakan Absolute path
    )


⏳ Menghubungkan ke server Cloudflare (Mohon tunggu sekitar 5 detik)...

🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵
🚀 MESIN IMAGEFORGE BERHASIL MENYALA!
👉 KLIK LINK INI UNTUK MEMBUKA APLIKASI: https://adam-nevertheless-ranks-wild.trycloudflare.com
🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


## 💖 Lisensi ImageForge: MIT License

ImageForge adalah proyek *open-source* yang tersedia di bawah **Lisensi MIT**. Ini berarti Anda memiliki kebebasan untuk menggunakan, memodifikasi, dan mendistribusikan kode ini untuk tujuan pribadi maupun komersial.

### ✨ Apa Artinya Bagi Anda?

*   **Gunakan Sesuka Hati**: Anda bebas menggunakan ImageForge dalam proyek Anda, tanpa batasan.
*   **Modifikasi**: Anda dapat mengubah kode sumber ImageForge sesuai kebutuhan Anda.
*   **Distribusi**: Anda dapat menyalin dan mendistribusikan ulang ImageForge, baik versi asli maupun yang sudah Anda modifikasi.
*   **Gratis**: Lisensi ini tidak membebankan biaya apapun.

### ⚖️ Kondisi Penting:

*   Lisensi MIT ini harus disertakan dalam semua salinan atau bagian substansial dari perangkat lunak.
*   Kami menyediakan perangkat lunak ini "APA ADANYA" tanpa jaminan apapun. Pengembang tidak bertanggung jawab atas kerusakan atau masalah yang mungkin timbul dari penggunaan perangkat lunak ini.

Ini adalah lisensi yang sangat permisif, mendorong kolaborasi dan inovasi. Jika Anda ingin membaca detail lengkapnya, Anda bisa melihat [Lisensi MIT resmi di GitHub](https://opensource.org/licenses/MIT).

---